In [1]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, lit

In [2]:
spark = SparkSession.builder \
        .appName("Complex_json") \
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/20 22:43:04 WARN Utils: Your hostname, ast-ubuntu, resolves to a loopback address: 127.0.1.1; using 192.168.1.7 instead (on interface wlp2s0)
26/11/20 22:43:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/20 22:43:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
file_path = "/home/akshay/Documents/Setups/1Projects/RetailAnalysis/misc/PySpark/data/medications.json"

In [4]:
# Read the multi-line JSON document and inspect its nested schema.
raw_df = spark.read.option("multiLine", "true").json(file_path)
raw_df.printSchema()
raw_df.show(truncate=False)



root
 |-- imaging: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- location: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- time: string (nullable = true)
 |-- labs: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- location: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- time: string (nullable = true)
 |-- medications: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- aceInhibitors: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- dose: string (nullable = true)
 |    |    |    |    |-- name: string (nullable = true)
 |    |    |    |    |-- pillCount: string (nullable = true)
 |    |    |    |    |-- refills: string (nullable = true)
 |    |    |    |    |-- route: string (nullable = true)
 |    |    |    |    |-- sig: string (nullable = tr

In [5]:
# Explode the lab and imaging arrays into one row per item.
labs_df = raw_df.select(explode(col("labs")).alias("lab")).select("lab.*")
imaging_df = raw_df.select(explode(col("imaging")).alias("image")).select("image.*")

In [6]:
print("Labs:")
labs_df.show(truncate=False)

Labs:
+-------------------+------------------+-------+
|location           |name              |time   |
+-------------------+------------------+-------+
|Main Hospital Lab  |Arterial Blood Gas|Today  |
|Primary Care Clinic|BMP               |Today  |
|Primary Care Clinic|BNP               |3 Weeks|
|Primary Care Clinic|BUN               |1 Year |
|Primary Care Clinic|Cardiac Enzymes   |Today  |
|Primary Care Clinic|CBC               |1 Year |
|Main Hospital Lab  |Creatinine        |1 Year |
|Primary Care Clinic|Electrolyte Panel |1 Year |
|Main Hospital Lab  |Glucose           |1 Year |
|Primary Care Clinic|PT/INR            |3 Weeks|
|Coumadin Clinic    |PTT               |3 Weeks|
|Primary Care Clinic|TSH               |1 Year |
+-------------------+------------------+-------+



In [7]:
print("Imaging:")
imaging_df.show(truncate=False)

Imaging:
+-----------------------+-----------+-----+
|location               |name       |time |
+-----------------------+-----------+-----+
|Main Hospital Radiology|Chest X-Ray|Today|
|Main Hospital Radiology|Chest X-Ray|Today|
|Main Hospital Radiology|Chest X-Ray|Today|
+-----------------------+-----------+-----+



In [8]:
# Each medication category is an array inside the medications array.
medication_groups_df = raw_df.select(
    explode(col("medications")).alias("medication_group")
)
medication_categories = (
    raw_df.schema["medications"].dataType.elementType.fieldNames()
)

medication_dfs = []
for category in medication_categories:
    category_df = (
        medication_groups_df
        .select(
            explode(col(f"medication_group.{category}")).alias("medication")
        )
        .select(lit(category).alias("category"), col("medication.*"))
    )
    medication_dfs.append(category_df)

medications_df = medication_dfs[0]
for category_df in medication_dfs[1:]:
    medications_df = medications_df.unionByName(category_df)



In [9]:
print("Medications:")
medications_df.show(truncate=False)

Medications:


+--------------+-----+---------------------+---------+--------+-----+----------+---------------------+
|category      |dose |name                 |pillCount|refills |route|sig       |strength             |
+--------------+-----+---------------------+---------+--------+-----+----------+---------------------+
|aceInhibitors |1 tab|lisinopril           |#90      |Refill 3|PO   |daily     |10 mg Tab            |
|antianginal   |1 tab|nitroglycerin        |#30      |Refill 1|SL   |q15min PRN|0.4 mg Sublingual Tab|
|anticoagulants|1 tab|warfarin sodium      |#90      |Refill 3|PO   |daily     |3 mg Tab             |
|betaBlocker   |1 tab|metoprolol tartrate  |#90      |Refill 3|PO   |daily     |25 mg Tab            |
|diuretic      |1 tab|furosemide           |#90      |Refill 3|PO   |daily     |40 mg Tab            |
|mineral       |1 tab|potassium chloride ER|#90      |Refill 3|PO   |daily     |10 mEq Tab           |
+--------------+-----+---------------------+---------+--------+-----+----

In [10]:
spark.stop()